# Chapter 5 — Trees and Graphs

*Built with [Claude](https://claude.ai) by Anthropic.*

Trees and graphs are the backbone of hierarchical data, dependency resolution, and network analysis — all common in data pipelines and ML workflows.

In [14]:
from collections import deque
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def make_tree(vals):
    """Build a tree from a level-order list (None = missing node)."""
    if not vals:
        return None
    root = TreeNode(vals[0])
    queue = deque([root])
    i = 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root):
    """Serialize tree to level-order list."""
    if not root:
        return []
    result, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node:
            result.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append(None)
    while result and result[-1] is None:
        result.pop()
    return result

def show_tree(root, prefix='', is_left=True):
    """Print tree in visual format."""
    if root is None:
        return
    show_tree(root.right, prefix + ('│   ' if is_left else '    '), False)
    print(prefix + ('└── ' if is_left else '┌── ') + str(root.val))
    show_tree(root.left, prefix + ('    ' if is_left else '│   '), True)

print('  Helpers loaded: TreeNode, make_tree, show_tree')

  Helpers loaded: TreeNode, make_tree, show_tree


---

## Part 1 — Tree DFS

**DS/ML connection:** Traversing decision trees, computing subtree statistics, dependency graph resolution — all DFS on trees. `sklearn`'s `DecisionTreeClassifier` stores nodes in arrays but the traversal logic is DFS.

In [15]:
# ── DS/ML PARALLEL: decision tree traversal ──
from sklearn.tree import DecisionTreeClassifier
import numpy as np

X = np.array([[1,2],[3,4],[5,6],[7,8]])
y = np.array([0,0,1,1])
clf = DecisionTreeClassifier(max_depth=2, random_state=0)
clf.fit(X, y)

# sklearn exposes the tree as arrays — children_left/right are the DFS edges
print('  tree node count:', clf.tree_.node_count)
print('  children_left: ', clf.tree_.children_left)
print('  children_right:', clf.tree_.children_right)
print('  (DFS traversal is just following these arrays)')

  tree node count: 3
  children_left:  [ 1 -1 -1]
  children_right: [ 2 -1 -1]
  (DFS traversal is just following these arrays)


### LC 104 — [Maximum Depth of Binary Tree](https://leetcode.com/problems/maximum-depth-of-binary-tree/)

**Problem:** Return the maximum depth (number of nodes along the longest root-to-leaf path).

**DS parallel:** Depth of a nested JSON, depth of a hierarchical category taxonomy, depth of a sklearn decision tree.

**Key insight:** `maxDepth(node) = 1 + max(maxDepth(left), maxDepth(right))`. Base case: `None` → 0.

In [16]:
# ── DS/ML APPROACH: sklearn tree depth ──
def max_depth_sklearn(clf):
    return clf.get_depth()

print(f"  sklearn tree depth: {max_depth_sklearn(clf)}")

# ── RAW PYTHON (interview answer) ──
def maxDepth(root: Optional[TreeNode]) -> int:
    if not root:
        return 0
    return 1 + max(maxDepth(root.left), maxDepth(root.right))

# Iterative (BFS) version
def maxDepth_iter(root: Optional[TreeNode]) -> int:
    if not root:
        return 0
    depth = 0
    queue = deque([root])
    while queue:
        depth += 1
        for _ in range(len(queue)):
            node = queue.popleft()
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)
    return depth

tree = make_tree([3, 9, 20, None, None, 15, 7])
print(f"  tree:")
show_tree(tree)
print(f"  maxDepth (recursive): {maxDepth(tree)}")
print(f"  maxDepth (iterative): {maxDepth_iter(tree)}")

  sklearn tree depth: 1
  tree:
│       ┌── 7
│   ┌── 20
│   │   └── 15
└── 3
    └── 9
  maxDepth (recursive): 3
  maxDepth (iterative): 3


In [17]:
# ── TRACE: LC 104 ──
def maxDepth_trace(root, depth=0, prefix='root'):
    indent = '  ' + '  ' * depth
    if not root:
        print(f"{indent}{prefix}: None → 0")
        return 0
    print(f"{indent}{prefix}: node={root.val}")
    L = maxDepth_trace(root.left,  depth + 1, 'left')
    R = maxDepth_trace(root.right, depth + 1, 'right')
    result = 1 + max(L, R)
    print(f"{indent}  → 1 + max({L},{R}) = {result}")
    return result

tree = make_tree([3, 9, 20, None, None, 15, 7])
print(f"  maxDepth trace:")
maxDepth_trace(tree)

  maxDepth trace:
  root: node=3
    left: node=9
      left: None → 0
      right: None → 0
      → 1 + max(0,0) = 1
    right: node=20
      left: node=15
        left: None → 0
        right: None → 0
        → 1 + max(0,0) = 1
      right: node=7
        left: None → 0
        right: None → 0
        → 1 + max(0,0) = 1
      → 1 + max(1,1) = 2
    → 1 + max(1,2) = 3


3

### LC 112 — [Path Sum](https://leetcode.com/problems/path-sum/)

**Problem:** Return `True` if any root-to-leaf path has node values summing to `targetSum`.

**DS parallel:** Checking if any decision-tree path from root to leaf satisfies a threshold condition on cumulative values.

**Key insight:** Subtract current node value from target as you go down. At a leaf, check if remaining equals zero.

In [18]:
# ── DS/ML APPROACH: enumerate all root-to-leaf paths ──
def all_paths(root):
    """Return all root-to-leaf paths."""
    if not root:
        return []
    if not root.left and not root.right:
        return [[root.val]]
    paths = []
    for path in all_paths(root.left) + all_paths(root.right):
        paths.append([root.val] + path)
    return paths

def hasPathSum_brute(root, targetSum):
    return any(sum(p) == targetSum for p in all_paths(root))

# ── RAW PYTHON (interview answer) ──
def hasPathSum(root: Optional[TreeNode], targetSum: int) -> bool:
    if not root:
        return False
    if not root.left and not root.right:   # leaf
        return root.val == targetSum
    remaining = targetSum - root.val
    return hasPathSum(root.left, remaining) or hasPathSum(root.right, remaining)

tree = make_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
print(f"  All paths: {all_paths(tree)}")
print(f"  hasPathSum(22) brute={hasPathSum_brute(tree, 22)}  recursive={hasPathSum(tree, 22)}")
print(f"  hasPathSum(999) brute={hasPathSum_brute(tree, 999)}  recursive={hasPathSum(tree, 999)}")

  All paths: [[5, 4, 11, 7], [5, 4, 11, 2], [5, 8, 13], [5, 8, 4, 1]]
  hasPathSum(22) brute=True  recursive=True
  hasPathSum(999) brute=False  recursive=False


In [19]:
# ── TRACE: LC 112 ──
def hasPathSum_trace(root, remaining, depth=0):
    indent = '  ' + '  ' * depth
    if not root:
        print(f"{indent}None → False")
        return False
    rem_after = remaining - root.val
    is_leaf = not root.left and not root.right
    print(f"{indent}node={root.val}  remaining={remaining}  rem_after={rem_after}  leaf={is_leaf}")
    if is_leaf:
        result = rem_after == 0
        print(f"{indent}→ {result}")
        return result
    L = hasPathSum_trace(root.left,  rem_after, depth + 1)
    if L:
        print(f"{indent}→ True (left path)")
        return True
    R = hasPathSum_trace(root.right, rem_after, depth + 1)
    print(f"{indent}→ {R}")
    return R

tree = make_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
hasPathSum_trace(tree, 22)

  node=5  remaining=22  rem_after=17  leaf=False
    node=4  remaining=17  rem_after=13  leaf=False
      node=11  remaining=13  rem_after=2  leaf=False
        node=7  remaining=2  rem_after=-5  leaf=True
        → False
        node=2  remaining=2  rem_after=0  leaf=True
        → True
      → True
    → True (left path)
  → True (left path)


True

---

## Part 2 — Tree BFS

**DS/ML connection:** Level-order traversal = processing data layer by layer. Used in hierarchical clustering evaluation, computing statistics at each tree depth, and any breadth-first pipeline stage.

### LC 102 — [Binary Tree Level Order Traversal](https://leetcode.com/problems/binary-tree-level-order-traversal/)

**Problem:** Return the level-order traversal as a list of lists (each inner list is one level).

**DS parallel:** Processing a decision tree level by level to extract features at each depth, or computing statistics per layer in a hierarchical model.

**Key insight:** Use a queue. At the start of each iteration, snapshot the queue size — that many nodes belong to the current level.

In [20]:
# ── DS/ML APPROACH: serialize tree and group by depth ──
def level_order_dfs(root):
    """DFS that records (val, depth) pairs, then groups."""
    pairs = []
    def dfs(node, d):
        if not node: return
        pairs.append((node.val, d))
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    dfs(root, 0)
    if not pairs: return []
    import itertools
    pairs.sort(key=lambda x: x[1])
    return [[v for v, _ in g] for _, g in itertools.groupby(pairs, key=lambda x: x[1])]

# ── RAW PYTHON (interview answer) ──
def levelOrder(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result = []
    queue = deque([root])
    while queue:
        level = []
        for _ in range(len(queue)):   # process exactly this level
            node = queue.popleft()
            level.append(node.val)
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)
        result.append(level)
    return result

tree = make_tree([3, 9, 20, None, None, 15, 7])
show_tree(tree)
print(f"  BFS (queue):  {levelOrder(tree)}")
print(f"  DFS (groups): {level_order_dfs(tree)}")

│       ┌── 7
│   ┌── 20
│   │   └── 15
└── 3
    └── 9
  BFS (queue):  [[3], [9, 20], [15, 7]]
  DFS (groups): [[3], [9, 20], [15, 7]]


In [21]:
# ── TRACE: LC 102 ──
def levelOrder_trace(root):
    if not root:
        return []
    result = []
    queue = deque([root])
    lvl = 0
    while queue:
        level = []
        size = len(queue)
        print(f"  level {lvl}: queue size={size}, processing:", end='')
        for _ in range(size):
            node = queue.popleft()
            level.append(node.val)
            print(f" {node.val}", end='')
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)
        result.append(level)
        lvl += 1
        print(f"  → level={level}")
    print(f"  result: {result}")

tree = make_tree([3, 9, 20, None, None, 15, 7])
levelOrder_trace(tree)

  level 0: queue size=1, processing: 3  → level=[3]
  level 1: queue size=2, processing: 9 20  → level=[9, 20]
  level 2: queue size=2, processing: 15 7  → level=[15, 7]
  result: [[3], [9, 20], [15, 7]]


### LC 199 — [Binary Tree Right Side View](https://leetcode.com/problems/binary-tree-right-side-view/)

**Problem:** Return the values visible when looking at the tree from the right side (the rightmost node at each level).

**DS parallel:** Extracting the "last" or "dominant" value at each hierarchy level — e.g., last decision rule at each depth.

**Key insight:** BFS level-order traversal — take the last element of each level.

In [22]:
# ── DS/ML APPROACH: level-order then take last per level ──
def rightSideView_pandas(root):
    levels = level_order_dfs(root)
    return [lvl[-1] for lvl in levels]

# ── RAW PYTHON (interview answer) ──
def rightSideView(root: Optional[TreeNode]) -> List[int]:
    if not root:
        return []
    result = []
    queue = deque([root])
    while queue:
        for i in range(len(queue)):
            node = queue.popleft()
            if i == 0:  # first popped = rightmost of this level
                result.append(node.val)
            if node.right: queue.append(node.right)
            if node.left:  queue.append(node.left)
    return result

tree = make_tree([1, 2, 3, None, 5, None, 4])
show_tree(tree)
print(f"  right side view (take-last): {rightSideView_pandas(tree)}")
print(f"  right side view (BFS):       {rightSideView(tree)}")

│       ┌── 4
│   ┌── 3
└── 1
    │   ┌── 5
    └── 2
  right side view (take-last): [1, 3, 4]
  right side view (BFS):       [1, 3, 4]


In [23]:
# ── TRACE: LC 199 — right-first BFS ──
def rightSideView_trace(root):
    if not root:
        return []
    result = []
    queue = deque([root])
    lvl = 0
    while queue:
        size = len(queue)
        print(f"  level {lvl}: ", end='')
        for i in range(size):
            node = queue.popleft()
            if i == 0:
                result.append(node.val)
                print(f"{node.val}* ", end='')   # * = visible
            else:
                print(f"{node.val}  ", end='')
            if node.right: queue.append(node.right)
            if node.left:  queue.append(node.left)
        print(f"→ visible={result[-1]}")
        lvl += 1
    print(f"  right side: {result}")

tree = make_tree([1, 2, 3, None, 5, None, 4])
rightSideView_trace(tree)

  level 0: 1* → visible=1
  level 1: 3* 2  → visible=3
  level 2: 4* 5  → visible=4
  right side: [1, 3, 4]


---

## Part 3 — Graph DFS / BFS

**DS/ML connection:** Graph algorithms underpin everything from connected-component labeling in images, to PageRank, to feature propagation in graph neural networks. Visited-set tracking is the key difference from tree traversal.

### LC 200 — [Number of Islands](https://leetcode.com/problems/number-of-islands/)

**Problem:** Given a grid of `'1'` (land) and `'0'` (water), count the number of islands (connected groups of `'1'`s).

**DS parallel:** Connected component labeling in an image or raster grid (equivalent to `scipy.ndimage.label`).

**Key insight:** For each unvisited `'1'`, run DFS to mark all connected land as visited. Count how many times you start a new DFS.

In [24]:
# ── DS/ML APPROACH: scipy connected components ──
import numpy as np
from scipy import ndimage

def count_islands_scipy(grid):
    arr = np.array([[int(c) for c in row] for row in grid])
    labeled, n = ndimage.label(arr)
    return n

# ── RAW PYTHON (interview answer) ──
def numIslands(grid) -> int:
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != '1':
            return
        grid[r][c] = '0'   # mark visited by sinking the island
        dfs(r+1, c); dfs(r-1, c); dfs(r, c+1); dfs(r, c-1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                dfs(r, c)
                count += 1
    return count

grid1 = [
    ['1','1','1','1','0'],
    ['1','1','0','1','0'],
    ['1','1','0','0','0'],
    ['0','0','0','0','0']
]
grid1_copy = [row[:] for row in grid1]
print(f"  grid:")
for row in grid1_copy:
    print('   ', ' '.join(row))
print(f"  scipy:  {count_islands_scipy(grid1_copy)}")
print(f"  dfs:    {numIslands(grid1)}")

  grid:
    1 1 1 1 0
    1 1 0 1 0
    1 1 0 0 0
    0 0 0 0 0
  scipy:  1
  dfs:    1


In [25]:
# ── TRACE: LC 200 ──
def numIslands_trace(orig_grid):
    grid   = [row[:] for row in orig_grid]   # working copy — mark visited as '0'
    labels = [row[:] for row in orig_grid]   # display copy — label each cell by island id
    rows, cols = len(grid), len(grid[0])
    count = 0

    def bfs(r, c, island_id):
        queue = deque([(r, c)])
        grid[r][c] = '0'                     # mark visited before entering queue
        labels[r][c] = str(island_id)
        while queue:
            cr, cc = queue.popleft()
            for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]:
                nr, nc = cr + dr, cc + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == '1':
                    grid[nr][nc] = '0'       # mark before enqueue — prevents re-visits
                    labels[nr][nc] = str(island_id)
                    queue.append((nr, nc))

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                count += 1
                print(f"  → Starting island {count} from ({r},{c})")
                bfs(r, c, count)

    print(f"  Final labeled grid:")
    for row in labels:
        print('   ', ' '.join(row))
    print(f"  Total islands: {count}")

grid2 = [
    ['1','1','0','0','0'],
    ['1','1','0','0','0'],
    ['0','0','1','0','0'],
    ['0','0','0','1','1']
]
numIslands_trace(grid2)

  → Starting island 1 from (0,0)
  → Starting island 2 from (2,2)
  → Starting island 3 from (3,3)
  Final labeled grid:
    1 1 0 0 0
    1 1 0 0 0
    0 0 2 0 0
    0 0 0 3 3
  Total islands: 3


### LC 133 — [Clone Graph](https://leetcode.com/problems/clone-graph/)

**Problem:** Deep-copy a connected undirected graph where each node has a value and a list of neighbors.

**DS parallel:** Deep-copying a computation graph or dependency DAG (e.g., cloning a PyTorch autograd graph).

**Key insight:** BFS/DFS with a `visited` dict mapping `original → clone`. When you encounter a neighbor, check the dict first — if already cloned, reuse; otherwise create and continue.

In [26]:
import copy

class GraphNode:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors or []

def make_graph(adj):
    """Build graph from adjacency list (1-indexed)."""
    nodes = {i: GraphNode(i) for i in range(1, len(adj) + 1)}
    for i, nbrs in enumerate(adj, 1):
        nodes[i].neighbors = [nodes[n] for n in nbrs]
    return nodes[1]

def graph_adj(node):
    """Serialize graph to adjacency dict."""
    seen = {}
    def dfs(n):
        if n.val in seen: return
        seen[n.val] = [nb.val for nb in n.neighbors]
        for nb in n.neighbors: dfs(nb)
    dfs(node)
    return dict(sorted(seen.items()))

# ── DS/ML APPROACH: deepcopy ──
def cloneGraph_deepcopy(node):
    return copy.deepcopy(node)

# ── RAW PYTHON (interview answer) ──
def cloneGraph(node):
    if not node:
        return None
    visited = {}  # original → clone
    queue = deque([node])
    visited[node] = GraphNode(node.val)
    while queue:
        curr = queue.popleft()
        for nbr in curr.neighbors:
            if nbr not in visited:
                visited[nbr] = GraphNode(nbr.val)
                queue.append(nbr)
            visited[curr].neighbors.append(visited[nbr])
    return visited[node]

g = make_graph([[2,4],[1,3],[2,4],[1,3]])
g_clone = cloneGraph(g)
print(f"  original graph: {graph_adj(g)}")
print(f"  cloned  graph:  {graph_adj(g_clone)}")
print(f"  same object?    {g is g_clone}")

  original graph: {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [1, 3]}
  cloned  graph:  {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [1, 3]}
  same object?    False


In [27]:
# ── TRACE: LC 133 ──
def cloneGraph_trace(node):
    if not node:
        return None
    visited = {}
    queue = deque([node])
    visited[node] = GraphNode(node.val)
    print(f"  Start: clone node {node.val}")
    while queue:
        curr = queue.popleft()
        print(f"  Processing node {curr.val}, neighbors={[n.val for n in curr.neighbors]}")
        for nbr in curr.neighbors:
            if nbr not in visited:
                visited[nbr] = GraphNode(nbr.val)
                queue.append(nbr)
                print(f"    → clone node {nbr.val} (new)")
            else:
                print(f"    → node {nbr.val} already cloned")
            visited[curr].neighbors.append(visited[nbr])
    print(f"  Clone adjacency: {graph_adj(visited[node])}")
    return visited[node]

g = make_graph([[2,4],[1,3],[2,4],[1,3]])
cloneGraph_trace(g)

  Start: clone node 1
  Processing node 1, neighbors=[2, 4]
    → clone node 2 (new)
    → clone node 4 (new)
  Processing node 2, neighbors=[1, 3]
    → node 1 already cloned
    → clone node 3 (new)
  Processing node 4, neighbors=[1, 3]
    → node 1 already cloned
    → node 3 already cloned
  Processing node 3, neighbors=[2, 4]
    → node 2 already cloned
    → node 4 already cloned
  Clone adjacency: {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [1, 3]}


---
## Part 4 — Topological Sort (DAGs)

**DS/ML connection:** Airflow, dbt, and MLflow all run a topological sort every time they schedule a pipeline. The orchestrator asks: "which tasks have no unmet dependencies right now?" — that is exactly Kahn's algorithm. A `CycleError` in dbt means topological sort failed.

In [28]:
# ── DS/ML PARALLEL: pipeline dependency resolution ──────────────────────────
# Simulating what Airflow does when it receives a DAG definition

pipeline = {
    'raw_data':  [],                    # no dependencies — runs first
    'clean':     ['raw_data'],
    'featurize': ['clean'],
    'train':     ['featurize'],
    'evaluate':  ['train'],
    'report':    ['evaluate'],
}

def resolve_pipeline(deps):
    remaining = {task: set(prereqs) for task, prereqs in deps.items()}
    order = []
    while remaining:
        ready = [t for t, prereqs in remaining.items() if not prereqs]
        if not ready:
            raise ValueError('Cycle detected — pipeline cannot be scheduled!')
        for task in sorted(ready):
            order.append(task)
            del remaining[task]
            for prereqs in remaining.values():
                prereqs.discard(task)
    return order

order = resolve_pipeline(pipeline)
print('Execution order:')
for i, task in enumerate(order, 1):
    print(f'  {i}. {task}')

print()
print('Add a cycle (evaluate -> clean) and retry:')
pipeline_with_cycle = {k: list(v) for k, v in pipeline.items()}
pipeline_with_cycle['evaluate'] = ['train', 'clean']
pipeline_with_cycle['clean']    = ['evaluate']
try:
    resolve_pipeline(pipeline_with_cycle)
except ValueError as e:
    print(f'  Error: {e}')

Execution order:
  1. raw_data
  2. clean
  3. featurize
  4. train
  5. evaluate
  6. report

Add a cycle (evaluate -> clean) and retry:
  Error: Cycle detected — pipeline cannot be scheduled!


### Under the hood: Kahn's algorithm

`resolve_pipeline` above is Kahn's algorithm in disguise. The formal version works on node indices:

1. Compute **in-degree** (number of incoming edges) for every node
2. Enqueue all nodes with in-degree 0 — they have no prerequisites
3. BFS: pop a node → append to result → decrement in-degree of its neighbors → enqueue any that hit 0
4. If `len(result) == n`: valid order. Otherwise: cycle.

The Airflow analogy maps directly:
- in-degree 0 = task with no unmet dependencies = ready to run
- decrementing in-degree = marking a completed task as done
- cycle = `CycleError`

---
### LC 207 — [Course Schedule](https://leetcode.com/problems/course-schedule/)

> You have `n` courses numbered `0` to `n-1`. `prerequisites[i] = [a, b]` means you must take `b` before `a`. Return `True` if it's possible to finish all courses.
>
> Example: `n=2, prerequisites=[[1,0]]` → `True` (take 0, then 1)

**DS parallel:** Can this Airflow DAG be executed without deadlock? A cycle means some tasks can never start — each waits on the other.

**Key insight:** Build the prerequisite graph, run Kahn's. If the result contains all `n` courses, no cycle exists. LC 210 (Course Schedule II) is identical — return `order` instead of `True`.

In [29]:
from collections import deque, defaultdict

# ── DS/ML APPROACH ───────────────────────────────────────────────────────────
# Dependency resolver — same logic as resolve_pipeline(), adapted to LC format
def canFinish_ds(numCourses, prerequisites):
    deps = defaultdict(set)
    for course, prereq in prerequisites:
        deps[course].add(prereq)
    remaining = {i: set(deps[i]) for i in range(numCourses)}
    completed = 0
    changed = True
    while changed:
        changed = False
        ready = [c for c, prereqs in remaining.items() if not prereqs]
        for c in ready:
            del remaining[c]
            for prereqs in remaining.values():
                prereqs.discard(c)
            completed += 1
            changed = True
    return completed == numCourses

# ── RAW PYTHON — Kahn's algorithm (interview answer) ─────────────────────────
# O(V + E) time, O(V + E) space
def canFinish(numCourses, prerequisites):
    graph = defaultdict(list)
    in_degree = [0] * numCourses

    for course, prereq in prerequisites:
        graph[prereq].append(course)   # prereq unlocks course
        in_degree[course] += 1

    queue = deque(i for i in range(numCourses) if in_degree[i] == 0)
    completed = 0

    while queue:
        node = queue.popleft()
        completed += 1
        for neighbor in graph[node]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)

    return completed == numCourses

# LC 210 extension: return the order instead of True/False
def findOrder(numCourses, prerequisites):
    graph = defaultdict(list)
    in_degree = [0] * numCourses
    for course, prereq in prerequisites:
        graph[prereq].append(course)
        in_degree[course] += 1
    queue = deque(i for i in range(numCourses) if in_degree[i] == 0)
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    return order if len(order) == numCourses else []

cases = [
    (2, [[1,0]],                   True,  'simple chain'),
    (2, [[1,0],[0,1]],             False, '2-node cycle'),
    (4, [[1,0],[2,0],[3,1],[3,2]], True,  'diamond DAG'),
    (3, [[0,1],[1,2],[2,0]],       False, '3-node cycle'),
]
print(f"{'case':<18} {'n':>3}  {'ds':>7}  {'kahn':>7}  {'expected':>9}  order (LC 210)")
print('-' * 72)
for n, prereqs, expected, label in cases:
    ds   = canFinish_ds(n, prereqs)
    kahn = canFinish(n, prereqs)
    ord_ = findOrder(n, prereqs)
    ok   = '✓' if kahn == expected else '✗'
    print(f'{label:<18} {n:>3}  {str(ds):>7}  {str(kahn):>7}  {str(expected):>9}  {str(ord_):<20}  {ok}')

case                 n       ds     kahn   expected  order (LC 210)
------------------------------------------------------------------------
simple chain         2     True     True       True  [0, 1]                ✓
2-node cycle         2    False    False      False  []                    ✓
diamond DAG          4     True     True       True  [0, 1, 2, 3]          ✓
3-node cycle         3    False    False      False  []                    ✓


In [30]:
# ── TRACE: Kahn's algorithm step by step ─────────────────────────────────────
# 4 courses: 0 unlocks 1 and 2; both 1 and 2 must complete before 3 (diamond)
numCourses = 4
prerequisites = [[1,0],[2,0],[3,1],[3,2]]

graph = defaultdict(list)
in_degree = [0] * numCourses
for course, prereq in prerequisites:
    graph[prereq].append(course)
    in_degree[course] += 1

print(f'Courses: {numCourses}   Prerequisites (course <- prereq): {prerequisites}')
print(f'Graph (node unlocks): {dict(graph)}')
print(f'Initial in-degrees:   {dict(enumerate(in_degree))}')
print()

queue = deque(i for i in range(numCourses) if in_degree[i] == 0)
order = []
step = 0

print(f"  {'step':>4}  {'process':>9}  {'in-degrees':>20}  {'queue':>12}  order")
print('  ' + '-' * 68)

while queue:
    node = queue.popleft()
    order.append(node)
    step += 1
    for neighbor in graph[node]:
        in_degree[neighbor] -= 1
        if in_degree[neighbor] == 0:
            queue.append(neighbor)
    print(f"  {step:>4}  course {node}   {str(dict(enumerate(in_degree))):>20}  {str(list(queue)):>12}  {order}")

print()
if len(order) == numCourses:
    print(f'All {numCourses} courses completed. Valid order: {order}')
else:
    stuck = set(range(numCourses)) - set(order)
    print(f'Cycle detected — could not schedule: {stuck}')

Courses: 4   Prerequisites (course <- prereq): [[1, 0], [2, 0], [3, 1], [3, 2]]
Graph (node unlocks): {0: [1, 2], 1: [3], 2: [3]}
Initial in-degrees:   {0: 0, 1: 1, 2: 1, 3: 2}

  step    process            in-degrees         queue  order
  --------------------------------------------------------------------
     1  course 0   {0: 0, 1: 0, 2: 0, 3: 2}        [1, 2]  [0]
     2  course 1   {0: 0, 1: 0, 2: 0, 3: 1}           [2]  [0, 1]
     3  course 2   {0: 0, 1: 0, 2: 0, 3: 0}           [3]  [0, 1, 2]
     4  course 3   {0: 0, 1: 0, 2: 0, 3: 0}            []  [0, 1, 2, 3]

All 4 courses completed. Valid order: [0, 1, 2, 3]
